# Auto-label plates on a free Colab GPU

Turns a folder of lifting clips into a **YOLO plate-detection dataset** — no hand-drawing boxes.

How it works: an **open-vocabulary** detector (YOLO-World) is told to find `"weight plate"` in
each frame. Boxes that pass simple sanity gates (round plate ⇒ roughly square, sane size) are
written as YOLO labels. You then fine-tune a small Apache-friendly detector on them (the labels
are the asset; YOLO-World is only the labeller).

**Honest scope:** this auto-labels **plates only**. Pose joints already come from the pose model —
they don't need labels. Eval ground truth (rep count, depth made/missed, lockout) is *human
judgment* — the very thing we measure — so that stays manual in `eval/groundtruth/` (see
`eval/README.md`). This notebook does the boring 80%: plate boxes at scale.

**Setup:** Runtime → Change runtime type → **GPU (T4)**, then run cells top to bottom.


## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install

In [ ]:
!pip -q install ultralytics

## 3. Get clips into `/content/clips`

Pick **one**: either gather CC-licensed clips here (fully cloud), or upload a zip of your own
`clips/` folder. Skip the other cell.

### 3a. (Option A) Gather CC-only clips right here

In [ ]:
# Fully-cloud gather: same CC-only + duration filter as tools/gather_clips.py.
!pip -q install yt-dlp
import subprocess, pathlib

QUERIES = ["squat form side view", "deadlift technique", "powerlifting meet squat"]
CC = "Creative Commons Attribution license (reuse allowed)"
pathlib.Path("/content/clips").mkdir(exist_ok=True)
for q in QUERIES:
    subprocess.run(
        ["yt-dlp", "--match-filter", f"license = '{CC}' & duration < 180",
         "--download-archive", "/content/clips/downloaded.txt", "--write-info-json",
         "-f", "mp4/best", "-o", "/content/clips/%(id)s.%(ext)s", f"ytsearch20:{q}"],
        check=False,
    )
print("clips:", [p.name for p in pathlib.Path("/content/clips").glob("*.mp4")][:5])

### 3b. (Option B) Upload a zip of your own clips

In [ ]:
from google.colab import files
import zipfile, pathlib

pathlib.Path("/content/clips").mkdir(exist_ok=True)
up = files.upload()  # choose clips.zip
for name in up:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as z:
            z.extractall("/content/clips")
print("clips:", [p.name for p in pathlib.Path("/content/clips").glob("*")][:5])

### 3c. (Optional) Cut LONG videos into per-set clips

Skip if your clips are already short single sets. Otherwise drop long videos into `/content/raw`
(the upload line is in the cell), and this cuts each lifting *set* into its own clip in
`/content/clips` — using **body motion only** (no plate detector needed). Then run step 4 as usual.


In [ ]:
# OPTIONAL: turn LONG videos (whole gym sessions) into short per-set clips.
# Long videos go in /content/raw -> short clips come out in /content/clips.
import cv2, numpy as np, subprocess
from pathlib import Path
from ultralytics import YOLO

RAW = Path("/content/raw"); RAW.mkdir(exist_ok=True)
OUTC = Path("/content/clips"); OUTC.mkdir(exist_ok=True)

# (optional) upload long videos now -- uncomment:
# from google.colab import files
# up = files.upload()
# for n in up: Path(n).rename(RAW / n)

STRIDE, IMGSZ, WIN_S = 5, 384, 1.0
MOVE_THRESH, MAX_GAP_S, MIN_SET_S, PAD_S = 0.25, 4.0, 2.0, 1.0
L_HIP, R_HIP, L_SH, R_SH = 11, 12, 5, 6
pose = YOLO("yolo11n-pose.pt")  # fast scanner (motion only)


def hip_signal(v):
    cap = cv2.VideoCapture(str(v)); fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    ts, hips, torsos, f = [], [], [], 0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        if f % STRIDE == 0:
            r = pose.predict(fr, imgsz=IMGSZ, verbose=False)[0]
            hy = ty = np.nan
            kp = r.keypoints
            if kp is not None and kp.conf is not None and len(kp.conf):
                p = int(kp.conf.sum(dim=1).argmax()); pts = kp.xy[p].cpu().numpy()
                h = (pts[L_HIP, 1] + pts[R_HIP, 1]) / 2; s = (pts[L_SH, 1] + pts[R_SH, 1]) / 2
                hy, ty = h, abs(h - s)
            ts.append(f / fps); hips.append(hy); torsos.append(ty)
        f += 1
    cap.release()
    sc = np.nanmedian(np.array(torsos, float)); sc = sc if np.isfinite(sc) and sc >= 1 else 1.0
    return np.array(ts, float), np.array(hips, float) / sc, fps


def intervals(active, ts, end_t):
    runs, i, n = [], 0, len(active)
    while i < n:
        if active[i]:
            j = i
            while j + 1 < n and active[j + 1]:
                j += 1
            runs.append([float(ts[i]), float(ts[j])]); i = j + 1
        else:
            i += 1
    if not runs:
        return []
    m = [runs[0]]
    for s, e in runs[1:]:
        if s - m[-1][1] < MAX_GAP_S:
            m[-1][1] = e
        else:
            m.append([s, e])
    out = []
    for s, e in m:
        if e - s < MIN_SET_S:
            continue
        out.append((max(0.0, s - PAD_S), min(end_t, e + PAD_S)))
    return out


VID = (".mp4", ".mov", ".mkv", ".webm", ".avi")
total = 0
for v in sorted(p for p in RAW.glob("*") if p.suffix.lower() in VID):
    ts, hip, fps = hip_signal(v)
    if not len(ts):
        continue
    win = max(1, int(WIN_S * fps / STRIDE)); act = np.zeros(len(hip), bool)
    for i in range(len(hip)):
        seg = hip[max(0, i - win // 2): i + win // 2 + 1]; seg = seg[np.isfinite(seg)]
        if seg.size >= 2 and seg.max() - seg.min() > MOVE_THRESH:
            act[i] = True
    sets = intervals(act, ts, ts[-1] + STRIDE / fps)
    for k, (s, e) in enumerate(sets, 1):
        o = OUTC / f"{v.stem}_set{k:02d}.mp4"
        subprocess.run(["ffmpeg", "-y", "-ss", f"{s:.2f}", "-i", str(v), "-t", f"{e-s:.2f}",
                        "-c", "copy", str(o)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print(f"  {o.name}: {s:.1f}-{e:.1f}s ({e-s:.1f}s)")
    total += len(sets)
    print(f"{v.name}: {len(sets)} sets")
print(f"\n{total} per-set clips -> /content/clips. Now run step 4 to label them.")

## 4. Auto-label plates → `/content/dataset`

Output is the **exact YOLO layout** `tools/train_plate.py` expects, so training is one step later.
Tighten the gates (`CONF`, `AR_*`, `H_*`) if the review grid in step 5 shows junk boxes.

In [ ]:
import cv2, numpy as np, random, csv
from pathlib import Path
from ultralytics import YOLOWorld

# ---- config (tune if review grid looks wrong) ----
CLIPS    = Path("/content/clips")
OUT      = Path("/content/dataset")
STRIDE   = 5                # sample every Nth frame (lifts are slow; no need for every frame)
CONF     = 0.25            # YOLO-World detection confidence
VAL_FRAC = 0.2
AR_MIN, AR_MAX = 0.6, 1.7  # plate is round -> bbox should be ~square
H_MIN, H_MAX   = 0.05, 0.35  # plate height as a fraction of frame height
PROMPTS  = ["weight plate", "barbell plate", "bumper plate"]  # open-vocabulary classes
VID_EXT  = (".mp4", ".mov", ".mkv", ".webm", ".avi")

random.seed(0)
for sub in ("images/train", "images/val", "labels/train", "labels/val"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)

model = YOLOWorld("yolov8s-worldv2.pt")  # open-vocab detector — used only to LABEL
model.set_classes(PROMPTS)


def label_clip(vid: Path) -> dict:
    cap = cv2.VideoCapture(str(vid))
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    sampled = kept = f = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if f % STRIDE == 0:
            sampled += 1
            res = model.predict(frame, conf=CONF, verbose=False)[0]
            lines = []
            for cx, cy, bw, bh in res.boxes.xywh.cpu().numpy():
                ar = bw / max(bh, 1e-6)
                if not (AR_MIN <= ar <= AR_MAX):      # reject non-square (not a round plate)
                    continue
                if not (H_MIN <= bh / H <= H_MAX):    # reject too-small / too-big
                    continue
                lines.append(f"0 {cx/W:.6f} {cy/H:.6f} {bw/W:.6f} {bh/H:.6f}")
            if lines:
                split = "val" if random.random() < VAL_FRAC else "train"
                name = f"{vid.stem}_{f:05d}"
                cv2.imwrite(str(OUT / f"images/{split}/{name}.jpg"), frame)
                (OUT / f"labels/{split}/{name}.txt").write_text("\n".join(lines) + "\n")
                kept += 1
        f += 1
    cap.release()
    return {"clip": vid.name, "w": W, "h": H, "fps": round(fps, 1),
            "sampled": sampled, "labeled": kept, "hit_rate": round(kept / max(sampled, 1), 2)}


clips = sorted(p for p in CLIPS.glob("*") if p.suffix.lower() in VID_EXT)
report = [label_clip(c) for c in clips]

# data.yaml so train_plate.py / ultralytics can train immediately
(OUT / "data.yaml").write_text(
    f"path: {OUT.as_posix()}\ntrain: images/train\nval: images/val\nnc: 1\nnames: [plate]\n"
)

# QC report: flag clips that are probably not worth keeping
with (OUT / "qc_report.csv").open("w", newline="") as fh:
    if report:
        w = csv.DictWriter(fh, fieldnames=list(report[0].keys()) + ["flag"])
        w.writeheader()
        for r in report:
            flags = []
            if r["hit_rate"] < 0.3:
                flags.append("low-plate")
            if r["fps"] and r["fps"] < 25:
                flags.append("low-fps")
            if r["h"] < 540:
                flags.append("low-res")
            row = {**r, "flag": "|".join(flags) or "ok"}
            w.writerow(row)
            print(row)

total = sum(r["labeled"] for r in report)
print(f"\nLabeled {total} frames from {len(clips)} clips -> {OUT}")
print("Check qc_report.csv: 'flag' marks clips to review or drop.")

## 5. Review grid (spot-check the auto-labels before you trust them)

In [ ]:
import matplotlib.pyplot as plt, cv2, random
from pathlib import Path

imgs = sorted((Path("/content/dataset") / "images/train").glob("*.jpg"))
random.shuffle(imgs)
sample = imgs[:9]
plt.figure(figsize=(12, 12))
for i, p in enumerate(sample):
    im = cv2.imread(str(p))[:, :, ::-1].copy()
    H, W = im.shape[:2]
    for ln in (Path("/content/dataset") / "labels/train" / (p.stem + ".txt")).read_text().splitlines():
        _, cx, cy, bw, bh = map(float, ln.split())
        x1, y1 = int((cx - bw / 2) * W), int((cy - bh / 2) * H)
        x2, y2 = int((cx + bw / 2) * W), int((cy + bh / 2) * H)
        cv2.rectangle(im, (x1, y1), (x2, y2), (255, 0, 255), 3)
    plt.subplot(3, 3, i + 1)
    plt.imshow(im)
    plt.axis("off")
plt.tight_layout()
plt.show()
print("Junk boxes? raise CONF or tighten AR/H gates in step 4, re-run. Looks good? download below.")

## 6. Download the dataset

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/dataset", "zip", "/content/dataset")
files.download("/content/dataset.zip")

## 7. Next

1. Unzip `dataset.zip` into the repo's `dataset/` folder.
2. Train the plate detector (this notebook's GPU, or locally):
   `python tools/train_plate.py`  → `runs/plate/weights/best.pt` → copy to `models/plate.pt`.
3. Run it: `python cli.py input/clip.mov --lift squat --plate yolo`.
4. Grow the **eval** set by hand (rep count + legal pass/fail) in `eval/groundtruth/` —
   that's the human bit the auto-labeller can't do, and it's what makes the accuracy number real.

The flywheel: gather (CC) → auto-label plates here → review → train → measure on eval → repeat.
